# Aggregate Option B results

Companion notebook to `scripts/aggregate_results.py`. It reuses the script's
parsing functions so the numbers shown here are guaranteed to match the
CSV/JSON/LaTeX exported by the CLI. Point `RESULTS_ROOT` at whatever results
directory you want to analyze.

In [ ]:
import json
import sys
from pathlib import Path
from IPython.display import display

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make the aggregator importable (assume we are at the repo root or in paper/).
ROOT = Path.cwd()
REPO = ROOT if (ROOT / "scripts").exists() else ROOT.parent
sys.path.insert(0, str(REPO))
from scripts.aggregate_results import (  # noqa: E402
    SUMMARY_COLUMNS,
    collect_root,
    to_flat_row,
)

sns.set_theme(style="whitegrid")

# === EDIT THIS === point at the results directory to analyze.
RESULTS_ROOT = (
    REPO / "out_experiments/option_b_abblation_20260901_032555/option_b_abblation"
)

In [ ]:
records = collect_root(RESULTS_ROOT)
flat = pd.DataFrame([to_flat_row(r) for r in records])
print(f"{len(records)} experiments, "
      f"{flat['complete'].sum()} complete, "
      f"{(~flat['complete']).sum()} incomplete")

# Human-friendly model labels (strip the org prefix and 'es' suffix noise).
def short_model(name: str) -> str:
    return name.replace("__", "/")

flat["model_short"] = flat["model"].map(short_model)

display_cols = ["model_short", "mode", "complete",
                "test_exact", "test_f1", "gold_exact", "gold_f1"]
flat[display_cols].round(4)

In [ ]:
# Test F1/Exact per model, zsl vs ft. Missing (incomplete) ft is left blank.
pv = flat.pivot_table(
    index="model_short",
    columns="mode",
    values=["test_exact", "test_f1"],
    aggfunc="first",
)
pv.round(4)

In [ ]:
# Bar chart: F1 improvement from fine-tuning per model.
df = flat[flat["complete"]].pivot_table(
    index="model_short", columns="mode", values="test_f1", aggfunc="first"
).dropna(subset=["zsl", "ft"])
df["delta_ft_minus_zsl"] = df["ft"] - df["zsl"]
df = df.sort_values("ft", ascending=False)

ax = df[["zsl", "ft"]].plot(kind="bar", figsize=(10, 5), rot=15)
ax.set_ylabel("Test F1")
ax.set_title("Fine-tuned vs zero-shot test F1 per model")
ax.legend(title="Mode")
plt.tight_layout()
plt.show()

df.round(4)

In [ ]:
# Gold-audit F1 vs test F1: do FT gains on the big test set carry to the
# 200-item gold-audit sample?
g = flat[flat["complete"]].copy()
g = g.dropna(subset=["test_f1", "gold_f1"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, mode in zip(axes, ["zsl", "ft"]):
    sub = g[g["mode"] == mode]
    sns.scatterplot(data=sub, x="test_f1", y="gold_f1",
                    hue="model_short", ax=ax, s=100)
    lim = (min(sub[["test_f1", "gold_f1"]].min()) - 0.05,
           max(sub[["test_f1", "gold_f1"]].max()) + 0.05)
    ax.plot(lim, lim, ls="--", color="gray", lw=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_title(f"Mode={mode}: gold vs test F1")
    ax.set_xlabel("Test F1"); ax.set_ylabel("Gold-audit F1")
plt.tight_layout()
plt.show()

In [ ]:
# Question-kind F1 breakdown (from metrics_by_question_type.csv).
qt_rows = []
for r in records:
    if not r["complete"]:
        continue
    for qr in r["question_types"]:
        qt_rows.append({
            "model": short_model(r["model"]),
            "mode": r["mode"],
            "kind": qr["kind"],
            "f1": qr["f1"],
        })
qdf = pd.DataFrame(qt_rows)
if not qdf.empty:
    # Focus on the fine-tuned models for kind analysis.
    pivot_ft = qdf[qdf["mode"] == "ft"].pivot_table(
        index="kind", columns="model", values="f1", aggfunc="first"
    )
    pivot_ft.T.plot(kind="bar", figsize=(11, 5), rot=15)
    plt.title("Fine-tuned F1 by question kind")
    plt.ylabel("F1"); plt.xlabel("Model")
    plt.legend(title="Question kind", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
    display(pivot_ft.round(4))

In [ ]:
# Optional: training curves for a fine-tuned model that has training_history.csv.
history_candidates = [r for r in records if r["mode"] == "ft" and r["training_history"]]
if history_candidates:
    pick = history_candidates[0]
    hist = pd.read_csv(pick["training_history"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist["epoch"], hist["train_loss"], marker="o", label="train")
    axes[0].plot(hist["epoch"], hist["eval_loss"], marker="o", label="eval")
    axes[0].set_title(f"{short_model(pick['model'])} loss")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
    axes[1].plot(hist["epoch"], hist["eval_f1"], marker="o", color="C2")
    axes[1].set_title("eval F1"); axes[1].set_xlabel("epoch"); axes[1].set_ylabel("F1")
    plt.tight_layout()
    plt.show()
else:
    print("No fine-tuned run with training_history.csv found.")

In [ ]:
# Export the same machine-readable table the CLI produces, for the agent/paper.
export_path = REPO / "paper" / "results_summary.csv"
flat.to_csv(export_path, index=False)
print(f"Saved {len(flat)} rows -> {export_path}")

json_path = REPO / "paper" / "results_summary.json"
json_path.write_text(
    json.dumps(records, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print(f"Saved JSON -> {json_path}")